# VTF repaired harness build

Builds the exact repaired VTF native-HTTP harness corresponding to commit `c4ad206c9e75e3c0e71b806e2b3d4fb30ea21d48` and receiver blob `a308627476b264b5df196b1f5a43e823ef4b1883` using Flutter 3.47.5.

Run **Runtime → Run all**. The last cell downloads the APK.


In [ ]:
# Install system packages
!apt-get update -qq
!apt-get install -y -qq git curl unzip xz-utils zip libglu1-mesa openjdk-17-jdk > /dev/null

!rm -rf /content/flutter /content/vtf_build
!git clone --depth 1 --branch 3.47.5 https://github.com/flutter/flutter.git /content/flutter

import os, subprocess
os.environ["PATH"] = "/content/flutter/bin:" + os.environ["PATH"]
print(subprocess.check_output(["flutter","--version"], text=True))


In [ ]:
import os, pathlib, subprocess, shutil, zipfile, urllib.request
sdk = pathlib.Path("/content/android-sdk")
sdk.mkdir(exist_ok=True)
os.environ["ANDROID_SDK_ROOT"] = str(sdk)
os.environ["ANDROID_HOME"] = str(sdk)
os.environ["PATH"] = f"{sdk}/cmdline-tools/latest/bin:{sdk}/platform-tools:" + os.environ["PATH"]
url = "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip"
zip_path = "/content/cmdline-tools.zip"
urllib.request.urlretrieve(url, zip_path)
tmp = pathlib.Path("/content/cmdline-tools-tmp")
if tmp.exists(): shutil.rmtree(tmp)
tmp.mkdir()
with zipfile.ZipFile(zip_path) as z: z.extractall(tmp)
latest = sdk / "cmdline-tools" / "latest"
latest.parent.mkdir(parents=True, exist_ok=True)
if latest.exists(): shutil.rmtree(latest)
shutil.move(str(tmp / "cmdline-tools"), str(latest))
subprocess.run("yes | sdkmanager --licenses >/dev/null", shell=True, check=False)
subprocess.run(["sdkmanager","platform-tools","platforms;android-35","build-tools;35.0.0"], check=True)
subprocess.run(["flutter","config","--android-sdk",str(sdk)], check=True)


In [ ]:
import pathlib, subprocess, shutil
root = pathlib.Path("/content/vtf-build-only")
if root.exists(): shutil.rmtree(root)
subprocess.run(["git","clone","https://github.com/daleblacky/vtf-build-only.git",str(root)], check=True)
blob = subprocess.check_output(["git","hash-object","lib/vtf_native_receiver.dart"], cwd=root, text=True).strip()
print("RECEIVER_BLOB=", blob)
assert blob == "a308627476b264b5df196b1f5a43e823ef4b1883", blob
subprocess.run(["flutter","pub","get"], cwd=root, check=True)
stage = pathlib.Path("/content/vtf_scaffold")
if stage.exists(): shutil.rmtree(stage)
subprocess.run(["flutter","create","--platforms=android","--org","com.vexlorgrid.vtftest.nativehttp","--project-name","vtf_native_http_harness",str(stage)], check=True)
shutil.move(str(stage/"android"), str(root/"android"))


In [ ]:
import subprocess, pathlib, hashlib, shutil
root = pathlib.Path("/content/vtf-build-only")
subprocess.run(["flutter","analyze","lib/vtf_native_receiver.dart","lib/vtf_device_test_main.dart"], cwd=root, check=True)
subprocess.run(["flutter","build","apk","--debug","-t","lib/vtf_device_test_main.dart"], cwd=root, check=True)
apk = root/"build/app/outputs/flutter-apk/app-debug.apk"
data = apk.read_bytes()
sha = hashlib.sha256(data).hexdigest()
out = pathlib.Path("/content/VTF_REPAIRED_C4AD206.apk")
shutil.copy2(apk, out)
print("APK_READY=", out)
print("APK_BYTES=", len(data))
print("APK_SHA256=", sha)


In [ ]:
from google.colab import files
files.download("/content/VTF_REPAIRED_C4AD206.apk")
